[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/07_overlap_robustness_analysis.ipynb)


# 07 - Overlap Robustness Analysis

## Research question

How does polyphonic recognition quality change as the number of simultaneously active instruments increases?

## Approach

This notebook loads the prediction artifact from a completed polyphonic experiment and computes exact-match, micro-F1, macro-F1, precision, and recall grouped by `k_active`.

## What is fixed

Unless a selected config says otherwise: MedleyDB instrument labels, the `largest_balanced` split, MERT-v1-95M, 5 s segments, validation-based model selection, and test metrics saved only after training.

## What is varied

The selected predictions file; grouping by exact active-instrument count and a compact `>=5` bucket.

## Expected interpretation

A monotonic drop with larger k suggests overlap difficulty; class-specific failures should be cross-checked with `per_class_metrics.csv` from notebook 06.


## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- `EXPERIMENT_GROUPS`, `DATASET_GROUPS`, `MIXTURE_GROUPS`, and `SELECTED_EXPERIMENTS` to run one config at a time.
- `REPLACE_EXISTING = True` only when intentionally overwriting a completed run.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR, RUN_ROOT / "configs"]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
from pathlib import Path
import os
import shlex
import subprocess
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR, RUN_ROOT / "configs"]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)


In [ ]:
EXPERIMENT_GROUPS = {
    "polyphonic_protocols": [
        "polyphonic_largest_balanced_synthetic_k_mert95_last_mean_mlp",
        "polyphonic_largest_balanced_same_song_same_time_mert95_last_mean_mlp",
        "polyphonic_largest_balanced_full_mix_mert95_last_mean_mlp",
    ],
}
SELECTED_EXPERIMENT_ID = "polyphonic_largest_balanced_synthetic_k_mert95_last_mean_mlp"
PREDICTIONS_CSV = RESULTS_DIR / SELECTED_EXPERIMENT_ID / "seed_42" / "latest" / "predictions.csv"
if not PREDICTIONS_CSV.exists():
    direct = RESULTS_DIR / SELECTED_EXPERIMENT_ID / "predictions.csv"
    PREDICTIONS_CSV = direct if direct.exists() else PREDICTIONS_CSV
print("Predictions:", PREDICTIONS_CSV)


In [ ]:
def label_columns(frame, prefix):
    return [column for column in frame.columns if column.startswith(prefix)]


def overlap_bucket(k):
    k = int(k)
    return str(k) if k < 5 else ">=5"


def grouped_overlap_metrics(predictions):
    true_cols = label_columns(predictions, "true_")
    pred_cols = ["pred_" + c.removeprefix("true_") for c in true_cols]
    rows = []
    for bucket, group in predictions.assign(k_bucket=predictions["k_active"].map(overlap_bucket)).groupby("k_bucket"):
        y_true = group[true_cols].to_numpy(dtype=int)
        y_pred = group[pred_cols].to_numpy(dtype=int)
        rows.append({
            "k_active": bucket,
            "n": len(group),
            "exact_match": float((y_true == y_pred).all(axis=1).mean()),
            "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "micro_precision": float(precision_score(y_true, y_pred, average="micro", zero_division=0)),
            "micro_recall": float(recall_score(y_true, y_pred, average="micro", zero_division=0)),
        })
    order = {"1": 1, "2": 2, "3": 3, "4": 4, ">=5": 5}
    return pd.DataFrame(rows).sort_values(by="k_active", key=lambda s: s.map(order))

if PREDICTIONS_CSV.exists():
    predictions = pd.read_csv(PREDICTIONS_CSV)
    summary = grouped_overlap_metrics(predictions)
    display(summary)
    out = PREDICTIONS_CSV.parent / "overlap_metrics_by_k.csv"
    summary.to_csv(out, index=False)
    print("Saved:", out)
else:
    print("No predictions found yet. Run notebook 06 first.")
